# BioSkills Lab — Chapter 7
## Biological Data as Matrices and PCA on TCGA
[![BioSkills Lab](https://img.shields.io/badge/BioSkills-Lab-38bdf8)](https://bioskillslab.dev)

> Free Colab tier compatible. Downloads ~50MB using the cBioPortal public API.

In [ ]:
!pip install -q pandas numpy scikit-learn matplotlib requests

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import requests
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
print('Libraries loaded')

## Download TCGA multi-cancer RNA-seq data via cBioPortal

We use the **cBioPortal public API** to fetch RNA-seq expression data for 5 cancer types. This is ~50MB total and fits comfortably in free Colab RAM.

Cancer types: BRCA (breast), LUAD (lung), KIRC (kidney), PRAD (prostate), COAD (colon)

In [ ]:
BASE = 'https://www.cbioportal.org/api'

# Studies to include (TCGA provisional, RNA-seq)
STUDIES = {
    'BRCA': 'brca_tcga',
    'LUAD': 'luad_tcga',
    'KIRC': 'kirc_tcga',
    'PRAD': 'prad_tcga',
    'COAD': 'coad_tcga'
}

def get_sample_ids(study_id, max_samples=200):
    r = requests.get(f'{BASE}/studies/{study_id}/samples', timeout=30)
    samples = [s['sampleId'] for s in r.json()]
    return samples[:max_samples]

def get_expression(study_id, sample_ids, profile_id):
    payload = {
        'sampleIds': sample_ids,
        'entrezGeneIds': [],  # empty = all genes
        'projection': 'SUMMARY'
    }
    r = requests.post(
        f'{BASE}/molecular-profiles/{profile_id}/molecular-data/fetch?projection=SUMMARY',
        json={'sampleIds': sample_ids},
        timeout=120
    )
    return r.json()

print('Testing cBioPortal API...')
r = requests.get(f'{BASE}/health', timeout=10)
print(f'Status: {r.status_code}')

In [ ]:
# Fetch expression data for each cancer type
# Using mrna_median_Zscores profile which is small and pre-normalised
all_frames = []

for cancer, study in STUDIES.items():
    print(f'Fetching {cancer}...')
    profile = f'{study}_mrna'
    samples = get_sample_ids(study, max_samples=150)
    
    r = requests.post(
        f'{BASE}/molecular-profiles/{profile}/molecular-data/fetch',
        json={'sampleIds': samples},
        params={'projection': 'SUMMARY'},
        timeout=120
    )
    
    if r.status_code != 200:
        print(f'  Skipping {cancer}: {r.status_code}')
        continue
    
    data = r.json()
    df = pd.DataFrame([
        {'sample': d['sampleId'], 'gene': d['entrezGeneId'], 'value': d['value']}
        for d in data if d['value'] is not None
    ])
    if df.empty:
        print(f'  No data for {cancer}')
        continue
    
    pivot = df.pivot(index='sample', columns='gene', values='value')
    pivot['cancer_type'] = cancer
    all_frames.append(pivot)
    print(f'  {cancer}: {pivot.shape[0]} samples, {pivot.shape[1]-1} genes')

if all_frames:
    expr = pd.concat(all_frames, axis=0)
    print(f'\nTotal: {expr.shape}')
else:
    print('No data fetched — using synthetic fallback')

In [ ]:
# If API failed, use synthetic TCGA-like data for demonstration
if not all_frames:
    print('Generating synthetic TCGA-like data for demonstration...')
    np.random.seed(42)
    n_per_type, n_genes = 150, 3000
    cancer_types = ['BRCA','LUAD','KIRC','PRAD','COAD']
    labels_list, blocks = [], []
    for i, ct in enumerate(cancer_types):
        X = np.random.randn(n_per_type, n_genes) * 0.5
        X[:, i*600:(i+1)*600] += 3  # cancer-specific signature
        blocks.append(X)
        labels_list.extend([ct] * n_per_type)
    X_all = np.vstack(blocks)
    expr = pd.DataFrame(X_all, columns=[f'gene_{i}' for i in range(n_genes)])
    expr['cancer_type'] = labels_list
    print(f'Synthetic data: {expr.shape}')

In [ ]:
# Extract X and y
y_raw = expr['cancer_type'].values
X_raw = expr.drop(columns=['cancer_type']).values.astype(np.float32)

# Fill NaN with column mean
col_means = np.nanmean(X_raw, axis=0)
nan_mask = np.isnan(X_raw)
X_raw[nan_mask] = np.take(col_means, np.where(nan_mask)[1])

print(f'Samples: {X_raw.shape[0]}, Features: {X_raw.shape[1]}')
print(f'Cancer types: {np.unique(y_raw)}')

In [ ]:
# Encode labels
le = LabelEncoder()
y_enc = le.fit_transform(y_raw)
print(f'Classes: {le.classes_}')

In [ ]:
# Train/test split, scale, PCA
X_train, X_test, y_train, y_test = train_test_split(
    X_raw, y_enc, test_size=0.2, random_state=42, stratify=y_enc
)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

n_comps = min(50, X_train_s.shape[1], X_train_s.shape[0]-1)
pca = PCA(n_components=n_comps)
X_train_pca = pca.fit_transform(X_train_s)
X_test_pca  = pca.transform(X_test_s)
print(f'PCA complete. Shape: {X_train_pca.shape}')

In [ ]:
var = pca.explained_variance_ratio_
plt.figure(figsize=(10,4))
plt.subplot(1,2,1)
plt.bar(range(1, len(var)+1), var*100)
plt.xlabel('PC'); plt.ylabel('Variance (%)')
plt.title('Scree Plot')
plt.subplot(1,2,2)
plt.plot(np.cumsum(var)*100)
plt.xlabel('Number of PCs'); plt.ylabel('Cumulative variance (%)')
plt.title('Cumulative Variance')
plt.axhline(80, color='red', linestyle='--', label='80%')
plt.legend()
plt.tight_layout(); plt.show()
print(f'Top 10 PCs explain: {sum(var[:10])*100:.1f}%')

In [ ]:
fig, ax = plt.subplots(figsize=(10,7))
colors = ['#f97316','#38bdf8','#4ade80','#a78bfa','#f43f5e']
for i, ct in enumerate(le.classes_):
    m = y_train == i
    ax.scatter(X_train_pca[m,0], X_train_pca[m,1],
               label=ct, alpha=0.6, s=20, color=colors[i % len(colors)])
ax.set_xlabel(f'PC1 ({var[0]*100:.1f}%)')
ax.set_ylabel(f'PC2 ({var[1]*100:.1f}%)')
ax.set_title('PCA of TCGA Pan-Cancer Gene Expression')
ax.legend(fontsize=9)
plt.tight_layout(); plt.show()